In [0]:
# ============================================================
# 00_explore_engine_mechanics.py
# Phase 1 — Distributed Compute Topology Demo
# ============================================================

# Load the dataset
df = spark.table("yellow_trip_data.yellow.yellow_tripdata_2024_01")

print(f"Total rows: {df.count()}")


In [0]:

# -----------------------------------------------------------
# DEMO 1: A simple transformation — NO shuffle
# -----------------------------------------------------------
filtered = df.filter(df.passenger_count > 2)
filtered.show(5)
# Go to Spark UI -> SQL/DataFrame tab -> click this query
# Notice: only ONE stage. No shuffle. Each worker filters its own partition independently.


In [0]:

# -----------------------------------------------------------
# DEMO 2: A groupBy — TRIGGERS a shuffle
# -----------------------------------------------------------
revenue_by_payment = df.groupBy("payment_type").sum("fare_amount")
revenue_by_payment.show()
# Go to Spark UI -> this query now has TWO stages
# Stage 1: each worker computes partial sums on its own partition
# Stage 2 (after SHUFFLE): partial sums are redistributed by payment_type
#          and combined into final totals
# Click on the query -> you'll see "Exchange" in the plan = the shuffle


In [0]:

# -----------------------------------------------------------
# DEMO 3: Driver OOM simulation (DON'T actually run on full data!)
# -----------------------------------------------------------
# This pulls ALL rows to the driver's memory — DANGEROUS on large data
# small_sample = df.limit(1000)
# pandas_df = small_sample.toPandas()   # SAFE - only 1000 rows
# 
# If you did df.toPandas() on the FULL 3 million rows, the driver
# (which has limited memory, e.g. 4-8GB on Community Edition)
# would try to hold the entire dataset -> Driver OOM

print("Driver OOM happens with .collect()/.toPandas() on large unfiltered data")
print("Worker OOM happens when groupBy/join keys are heavily skewed")


In [0]:

# -----------------------------------------------------------
# DEMO 4: Worker OOM / Skew demonstration (conceptual + real)
# -----------------------------------------------------------
df.groupBy("payment_type").count().show()
# Notice payment_type=1 (Credit Card) has WAY more rows than payment_type=4
# If this dataset were 100x bigger, the worker handling "Credit Card"
# partition after shuffle would receive a disproportionate amount of data
# -> that worker's executor could OOM while others sit idle